# Spellchecking (Tolerant retrieval)

**Information Retrieval — Lab 04**

Have you ever googled a name without knowing exactly how it is spelled? Or typed a query in a rush and still got what you wanted? Modern search engines cope with defective input remarkably well, and the machinery behind that is **tolerant retrieval**. In this lab we take one more step towards a real search engine and handle two situations:

1. **The user knows they don't know the spelling**, or wants every word that follows a pattern, so they use a *wildcard* query such as `retr*val`.
2. **The user makes a mistake** (or doesn't care, or is in a hurry) and we have to repair it. We compare three classic approaches:
   1. a simple statistical spellchecker by Peter Norvig;
   2. phonetic correction with the Soundex algorithm;
   3. character trigrams ranked with the Jaccard coefficient.

**Reading.** Manning, Raghavan, Schütze, [*Introduction to Information Retrieval*](https://nlp.stanford.edu/IR-book/pdf/irbookonlinereading.pdf), chapter 3 (sections 3.2.2 wildcards, 3.3 spelling correction, 3.4 Soundex).

**Requirements.** Python ≥ 3.8 and the standard library only. Datasets are downloaded automatically on first run and cached in `./data/`. Expect the first run to take a few minutes (the typo corpus is large); later runs reuse the cache.

## 0. Setup

In [1]:
!uv tree -d 1

Resolved 128 packages in 10ms
ir v0.1.0
├── beautifulsoup4 v4.15.0
├── bs4 v0.0.2
├── feedparser v6.0.14
├── ipykernel v7.3.0
├── matplotlib v3.10.9
├── nltk v3.10.3
├── pandas v2.3.3
├── requests v2.34.2
├── selenium v4.48.0
└── statsmodels v0.15.0


In [2]:
import html
import random
import re
import string
import time
import urllib.request
from collections import Counter, defaultdict
from pathlib import Path

try:
    from IPython.display import HTML, display
except ImportError:                       # lets the code also run as a plain script
    HTML = None
    def display(x):
        print(x)

SEED = 42                                 # fixed seed -> reproducible samples
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

FACTS_URL = "https://raw.githubusercontent.com/hsu-ai-course/hsu.ai/master/code/datasets/nlp/facts.txt"
# TYPO_URL = "https://github-typo-corpus.s3.amazonaws.com/data/github-typo-corpus.v1.0.0.jsonl.gz"
# TYPO_URL = "https://github-typo-corpus.s3.amazonaws.com/data/github-typo-corpus.v1.0.0.jsonl.gz"
BIG_URLS = ["https://norvig.com/big.txt", "https://www.norvig.com/big.txt"]
TESTSET_URLS = {
    "development": ["https://norvig.com/spell-testset1.txt"], #, "https://www.norvig.com/spell-testset1.txt"],
    "final":       ["https://norvig.com/spell-testset2.txt"], #"https://www.norvig.com/spell-testset2.txt"],
}


def download(urls, path):
    """Download the first working URL in `urls` to `path`, unless it is already cached."""
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        return path
    urls = [urls] if isinstance(urls, str) else list(urls)
    errors = []
    for url in urls:
        print(f"Downloading {url} ...")
        tmp = path.with_name(path.name + ".part")
        try:
            urllib.request.urlretrieve(url, tmp)
            tmp.rename(path)
            return path
        except Exception as e:
            tmp.unlink(missing_ok=True)
            errors.append(f"{url}: {e}")
    raise RuntimeError("Could not download the file:\n  " + "\n  ".join(errors)
                       + f"\nDownload it manually and place it at '{path}'.")



def show_html(markup):
    """Render HTML in Jupyter; fall back to plain text elsewhere."""
    if HTML is not None:
        display(HTML(markup))
    else:
        print(re.sub(r"<[^>]+>", "", markup))

## 1. Handling wildcards

We handle wildcard queries with **k-grams**: all consecutive substrings of `k` characters of a word padded with a boundary symbol `$`. For *star* and `k = 3` the padded word is `$star$` and the 3-grams are `$st`, `sta`, `tar`, `ar$`. The idea (book, section 3.2.2): a query like `re*ed` contains the k-grams `$re` and `ed$`; every matching word must contain both, so **intersecting the postings** of these k-grams gives a small candidate set. Since the intersection can contain false positives (for the query red*, $re AND red to the 3-gram index, retired), we finish with a regular-expression check (post-filtering step).

Here we consider only the `*` wildcard (it may occur several times).

> **Why a new index?** The inverted index from the previous lab stores *stemmed* terms. A wildcard is matched against *surface forms* (`retr*val` must find `retrieval`, not `retriev`), so we need a vocabulary of original word forms.

### What you implement

| Function | Purpose |
|---|---|
| `build_inverted_index_orig_forms` | inverted index over lowercased, unstemmed tokens. Format: `term: [collection_frequency, (doc_id, doc_freq), ...]` |
| `build_k_gram_index` | maps each k-gram to the list of vocabulary words containing it. Format: `'k_gram': ['word1', 'word2', ...]` |
| `generate_wildcard_options` | vocabulary words matching a wildcard (k-gram intersection + regex filter) |
| `search_wildcard` | documents (facts) that contain at least one matching word |

We test on a small collection of curious facts.

In [3]:
facts_path = download(FACTS_URL, DATA_DIR / "facts.txt")
with open(facts_path, encoding="utf-8") as fp:
    facts = [line.strip() for line in fp if line.strip()]

print(f"{len(facts)} facts. The last five:")
print(*facts[-5:], sep="\n")

159 facts. The last five:
151. Women have twice as many pain receptors on their body than men. But a much higher pain tolerance.
152. There are more stars in space than there are grains of sand on every beach in the world.
153. For every human on Earth there are 1.6 million ants.
154. The total weight of all those ants, however, is about the same as all the humans.
155. On Jupiter and Saturn it rains diamonds.


In [ ]:
TOKEN_RE = re.compile(r"\w+(?:'\w+)*")      # words and numbers, keeps inner apostrophes (don't)
# ['don', 't', 'panic']
# ["don't", 'panic']

def tokenize(text):
    """Lowercase `text` and split it into word tokens (punctuation is dropped)."""
    return TOKEN_RE.findall(text.lower())


def build_inverted_index_orig_forms(documents):
    """Inverted index of original word forms (lowercased, no stemming).

    Returns {term: [collection_frequency, (doc_id, doc_freq), ...]}.
    """
    inverted_index = {}
    # TODO
    return inverted_index


def split_to_k_grams(word, k):
    """All consecutive substrings of length k (with repetitions)."""
    return [word[i:i + k] for i in range(len(word) - k + 1)]


def build_k_gram_index(vocabulary, k):
    """Map every k-gram to the list of words containing it.

    `vocabulary` is any iterable of words (a dict/Counter is fine: only the keys are used).
    Words are padded as '$word$' before splitting, so that word boundaries are k-grams too.
    """
    k_gram_index = defaultdict(list)

    # TODO
    
    return dict(k_gram_index)


def get_k(k_gram_index):
    """Recover k from an existing k-gram index."""
    return len(next(iter(k_gram_index)))


def wildcard_to_regex(wildcard):
    """'re*ed' -> compiled regex that must match a *whole* word."""
    return re.compile(".*".join(re.escape(part) for part in wildcard.split("*")))


def wildcard_candidates(wildcard, k_gram_index):
    """Words sharing all k-grams of the wildcard (may contain false positives).

    Returns None when the wildcard has no usable k-gram (e.g. 'a*').
    """
    k = get_k(k_gram_index)
    grams = set()

    # TODO

    if not grams:
        return None
    
    # intersect starting from the shortest postings list (cheapest first)
    # To intersect postings lists A, B, C, the final answer has at most min(|A|, |B|, |C|)
    postings = sorted((k_gram_index.get(g, []) for g in grams), key=len)
    candidates = set(postings[0])

    # TODO
    
    return candidates


def generate_wildcard_options(wildcard, k_gram_index, inverted_index):
    """All vocabulary words matching `wildcard`, most frequent first."""

    # TODO

    return []


def search_wildcard(wildcard, k_gram_index, index, docs):
    """Documents (facts) containing at least one word that matches the wildcard."""
    # TODO

    return []

### 1.1 A closer look at the k-gram filter

A toy example shows why the final regex check is needed. The wildcard `re*ed` produces the k-grams `$re` and `ed$`. The word *red* contains both (`$re`, `red`, `ed$`) although the two parts would have to overlap, so it is a **false positive** of the intersection step.

In [5]:
print("3-grams of 'star':", split_to_k_grams("$star$", 3))

toy_docs = ["The red reed was retired", "She agreed to proceed", "A tired reader reviewed it"]
toy_index = build_inverted_index_orig_forms(toy_docs)
toy_kgrams = build_k_gram_index(toy_index, 3)

print("after k-gram intersection:", sorted(wildcard_candidates("re*ed", toy_kgrams)))
print("after regex filtering    :", sorted(generate_wildcard_options("re*ed", toy_kgrams, toy_index)))

assert "red" in wildcard_candidates("re*ed", toy_kgrams)
assert "red" not in generate_wildcard_options("re*ed", toy_kgrams, toy_index)
assert generate_wildcard_options("re*ed", toy_kgrams, toy_index) == ["reed", "retired", "reviewed"]

3-grams of 'star': ['$st', 'sta', 'tar', 'ar$']
after k-gram intersection: ['red', 'reed', 'retired', 'reviewed']
after regex filtering    : ['reed', 'retired', 'reviewed']


### 1.2 Tests

In [6]:
index_orig_forms = build_inverted_index_orig_forms(facts)
k_gram_index = build_k_gram_index(index_orig_forms, 3)
print(f"vocabulary: {len(index_orig_forms)} words, k-gram index: {len(k_gram_index)} trigrams")

wildcard = "re*ed"
# wildcard = "re*ed"
wildcard_options = generate_wildcard_options(wildcard, k_gram_index, index_orig_forms)
print("options:", wildcard_options)
assert len(wildcard_options) >= 3

wildcard_results = search_wildcard(wildcard, k_gram_index, index_orig_forms, facts)


def highlight(text, terms):
    """HTML-escape `text` and mark every whole-word occurrence of `terms`."""
    if not terms:
        return html.escape(text)
    pattern = re.compile(r"\b(" + "|".join(map(re.escape, terms)) + r")\b", re.I)
    out, last = [], 0
    for m in pattern.finditer(text):
        out.append(html.escape(text[last:m.start()]))
        out.append(f"<mark><b>{html.escape(m.group(0))}</b></mark>")
        last = m.end()
    out.append(html.escape(text[last:]))
    return "".join(out)


show_html("<br>".join(highlight(r, wildcard_options) for r in wildcard_results))
assert len(wildcard_results) >= 3

assert "13. James Buchanan, the 15th U.S. president continuously bought slaves with his own money in order to free them." in search_wildcard("pres*dent", k_gram_index, index_orig_forms, facts)
assert "40. 9 out of 10 Americans are deficient in Potassium." in search_wildcard("p*tas*um", k_gram_index, index_orig_forms, facts)
assert "61. A man from Britain changed his name to Tim Pppppppppprice to make it harder for telemarketers to pronounce." in search_wildcard("*price", k_gram_index, index_orig_forms, facts)

# edge cases: no star, wildcard too short for any k-gram, upper case input
assert generate_wildcard_options("price", k_gram_index, index_orig_forms) == ["price"] or "price" not in index_orig_forms
assert set(generate_wildcard_options("a*", k_gram_index, index_orig_forms)) == {w for w in index_orig_forms if w.startswith("a")}
assert generate_wildcard_options("PRES*DENT", k_gram_index, index_orig_forms) == generate_wildcard_options("pres*dent", k_gram_index, index_orig_forms)
print("All wildcard tests passed.")

vocabulary: 1070 words, k-gram index: 2011 trigrams
options: ['reduced', 'received', 'recorded']


All wildcard tests passed.


In [7]:
wildcard = "re*ve*"
wildcard_options = generate_wildcard_options(wildcard, k_gram_index, index_orig_forms)
print("options:", wildcard_options)

wildcard_results = search_wildcard(wildcard, k_gram_index, index_orig_forms, facts)

show_html("<br>".join(highlight(r, wildcard_options) for r in wildcard_results))

options: ['received', 'restrictive']


**Think about it.**

- What happens to the index size when you change `k` from 2 to 4? Which `k` makes `*` queries cheaper, and for which queries does a large `k` stop helping?
- Why do we pad with `$`? What would `*ing` return without padding?
- A query like `*a*` has no usable k-gram. What does our implementation do, and what would this cost on a vocabulary of 10 million terms?

## 2. Handling typos

### 2.1 Data: training text and test sets

We use the classic resources from Peter Norvig's [essay](https://norvig.com/spell-correct.html):

- [`big.txt`](https://norvig.com/big.txt): about a million words of running English (public-domain books from Project Gutenberg plus word-frequency lists). It gives us the **vocabulary** of correct words and their **frequencies**.
- **Two test sets** of `right: wrong1 wrong2 ...` lines extracted from Roger Mitton's *Birkbeck spelling error corpus* of real human misspellings: a **development** set (270 pairs) and a **final** set (400 pairs).

Because the vocabulary and the test sets come from different sources, the evaluation is *open-vocabulary*: some intended words are simply missing from `big.txt`, and no dictionary-based method can recover them. We measure that ceiling explicitly below.

In [8]:
big_path = download(BIG_URLS, DATA_DIR / "big.txt")
with open(big_path, encoding="utf-8", errors="ignore") as fp:
    big_text = fp.read()
print(f"big.txt: {len(big_text) / 1e6:.1f} MB of text")


def load_testset(name):
    """[(right, wrong), ...] from a Norvig test file with lines 'right: wrong1 wrong2 ...'."""
    try:
        path = download(TESTSET_URLS[name], DATA_DIR / f"spell-testset-{name}.txt")
    except RuntimeError as e:
        print(f"WARNING: test set '{name}' is not available and will be skipped.\n{e}")
        return []
    pairs = []
    with open(path, encoding="utf-8") as fp:
        for line in fp:
            if ":" not in line:
                continue
            right, wrongs = line.split(":", 1)
            pairs += [(right.strip().lower(), wrong.lower()) for wrong in wrongs.split()]
    return pairs


testsets = {name: load_testset(name) for name in TESTSET_URLS}
testsets = {name: pairs for name, pairs in testsets.items() if pairs}
assert testsets, "no test set could be loaded"
for name, pairs in testsets.items():
    print(f"{name:<12} set: {len(pairs)} (right, wrong) pairs")

big.txt: 6.5 MB of text
development  set: 270 (right, wrong) pairs
final        set: 400 (right, wrong) pairs


#### Explore sample typos

Take a moment to browse the data. Note that the errors are of different kinds: omitted or doubled letters (`begining`), transpositions (`recieve`), phonetic spellings (`discription`) and sometimes **real words** (`where` for *were*), which a dictionary-based method cannot detect.

In [9]:
all_pairs = [p for pairs in testsets.values() for p in pairs]
for right, wrong in random.Random(SEED).sample(all_pairs, 10):
    print(f"{wrong:<18} => {right}")

reequired          => required
specail            => special
arrainged          => arranged
creadit            => credit
oppossite          => opposite
astablishing       => establishing
arragment          => arrangement
spledid            => splendid
carear             => career
vistid             => visited


#### Build a vocabulary

We need a dictionary of correct words for Norvig's spellchecker as well as for the Soundex and k-gram indexes. We take all alphabetic words of `big.txt`, lowercased, together with their counts. Restricting to letters filters out numbers, punctuation and markup.


In [10]:
WORD_RE = re.compile(r"[^\W\d_]+")          # runs of letters only (no digits, no underscores)


def sent_to_words(sent):
    """Split a text into lowercase alphabetic words."""
    return WORD_RE.findall(sent.lower())


vocabulary = Counter(sent_to_words(big_text))
print("vocabulary size:", len(vocabulary), "| tokens:", sum(vocabulary.values()))
print("most common:", vocabulary.most_common(10))

vocabulary size: 29157 | tokens: 1105285
most common: [('the', 80030), ('of', 40025), ('and', 38313), ('to', 28766), ('in', 22050), ('a', 21155), ('that', 12512), ('he', 12401), ('was', 11410), ('it', 10681)]


### 2.2 Context-independent spellcheckers

We implement, in order:

0. the **edit distance**;
1. **Norvig's** corrector ([essay](https://norvig.com/spell-correct.html));
2. **Soundex** ([Wikipedia](https://en.wikipedia.org/wiki/Soundex), book section 3.4);
3. **trigrams with the Jaccard coefficient**.

All of them work on a single word and ignore its context; that limitation is discussed at the end.

#### Edit distance

The Damerau–Levenshtein distance (the *optimal string alignment* variant) is the minimum number of insertions, deletions, substitutions and **transpositions of two adjacent characters** needed to turn one string into the other.

In [11]:
def edit_dist(s1, s2) -> int:
    """Damerau-Levenshtein (optimal string alignment) distance between s1 and s2."""
    if len(s1) < len(s2):                      # keep the rows as short as possible
        s1, s2 = s2, s1
    prev2, prev = None, list(range(len(s2) + 1))
    for i in range(1, len(s1) + 1):
        cur = [i] + [0] * len(s2)
        for j in range(1, len(s2) + 1):
            cost = 0 if s1[i - 1] == s2[j - 1] else 1
            cur[j] = min(prev[j] + 1,          # deletion
                         cur[j - 1] + 1,       # insertion
                         prev[j - 1] + cost)   # substitution
            if i > 1 and j > 1 and s1[i - 1] == s2[j - 2] and s1[i - 2] == s2[j - 1]:
                cur[j] = min(cur[j], prev2[j - 2] + 1)    # transposition
        prev2, prev = prev, cur
    return prev[-1]

#### Tests

In [12]:
assert edit_dist("korrectud", "corrected") == 2, "Edit distance is computed incorrectly"
assert edit_dist("soem", "some") == 1, "Edit distance is computed incorrectly"
assert edit_dist("one", "one") == 0, "Edit distance is computed incorrectly"
assert edit_dist("", "abc") == 3 and edit_dist("abc", "") == 3
assert edit_dist("ab", "ba") == 1 and edit_dist("kitten", "sitting") == 3
assert edit_dist("abc", "xyz") == edit_dist("xyz", "abc") == 3     # symmetric
print("Edit distance tests passed.")

Edit distance tests passed.


#### Norvig's spellchecker

The idea is probabilistic: among all dictionary words close to the typo, choose the one with the highest prior probability (word frequency). Candidates are generated in tiers: the word itself if known, else known words at **edit distance 1**, else at **distance 2**, else the word unchanged. Distance 1 means one deletion, transposition, replacement or insertion.

In [ ]:
WORDS = vocabulary
N_WORDS = sum(WORDS.values())


def P(word, N=N_WORDS):
    "Probability of `word`."
    return WORDS[word] / N


def known(words):
    "The subset of `words` that appear in the dictionary of WORDS."
    return {w for w in words if w in WORDS}


def edits1(word):
    "All edits that are one edit away from `word`."
    letters = string.ascii_lowercase
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]

    # TODO
    deletes = []
    transposes = []
    replaces = []
    inserts = []
    # TODO
    
    return set(deletes + transposes + replaces + inserts)


def edits2(word):
    "All edits that are two edits away from `word` (a lazy generator)."
    return (e2 for e1 in edits1(word) for e2 in edits1(e1))


def candidates(word):
    "Generate possible spelling corrections for word."
    return known([word]) or known(edits1(word)) or known(edits2(word)) or {word}


def fix_typo_norvig(word) -> str:
    "Most probable spelling correction for word."
    return max(candidates(word), key=P)


def suggest_norvig(word, n=5) -> list:
    "Up to n best suggestions (all from the closest non-empty tier), most probable first."
    return sorted(candidates(word), key=lambda w: (-WORDS.get(w, 0), w))[:n]

#### Tests

In [28]:
fix_typo_norvig("korrectud")

'corrected'

In [29]:
known(["korrectud"])

set()

In [36]:
edits1("korrectud"),len(list(edits1("korrectud")))

({'akorrectud',
  'aorrectud',
  'bkorrectud',
  'borrectud',
  'ckorrectud',
  'correctud',
  'dkorrectud',
  'dorrectud',
  'ekorrectud',
  'eorrectud',
  'fkorrectud',
  'forrectud',
  'gkorrectud',
  'gorrectud',
  'hkorrectud',
  'horrectud',
  'ikorrectud',
  'iorrectud',
  'jkorrectud',
  'jorrectud',
  'kaorrectud',
  'karrectud',
  'kborrectud',
  'kbrrectud',
  'kcorrectud',
  'kcrrectud',
  'kdorrectud',
  'kdrrectud',
  'keorrectud',
  'kerrectud',
  'kforrectud',
  'kfrrectud',
  'kgorrectud',
  'kgrrectud',
  'khorrectud',
  'khrrectud',
  'kiorrectud',
  'kirrectud',
  'kjorrectud',
  'kjrrectud',
  'kkorrectud',
  'kkrrectud',
  'klorrectud',
  'klrrectud',
  'kmorrectud',
  'kmrrectud',
  'knorrectud',
  'knrrectud',
  'koarectud',
  'koarrectud',
  'kobrectud',
  'kobrrectud',
  'kocrectud',
  'kocrrectud',
  'kodrectud',
  'kodrrectud',
  'koerectud',
  'koerrectud',
  'kofrectud',
  'kofrrectud',
  'kogrectud',
  'kogrrectud',
  'kohrectud',
  'kohrrectud',
  'koire

In [34]:
list(edits2("korrectud")), len(list(edits2("korrectud")))

(['korrectuy',
  'okorrectuy',
  'koriectuy',
  'korractuy',
  'korrectumy',
  'bkorrectuy',
  'korreetuy',
  'korrectfuy',
  'koprrectuy',
  'korrdctuy',
  'korrecthuy',
  'korrectkuy',
  'korrectuv',
  'korretctuy',
  'korrwectuy',
  'korrecwtuy',
  'korrecyuy',
  'uorrectuy',
  'korrehctuy',
  'korrecfuy',
  'korrectly',
  'zkorrectuy',
  'kormectuy',
  'tkorrectuy',
  'correctuy',
  'kowrrectuy',
  'korrecteuy',
  'kokrectuy',
  'korreituy',
  'korrectur',
  'korrectuxy',
  'konrectuy',
  'kmrrectuy',
  'korrectud',
  'forrectuy',
  'oorrectuy',
  'korrbctuy',
  'korbrectuy',
  'ktorrectuy',
  'korrectruy',
  'kwrrectuy',
  'korrecty',
  'borrectuy',
  'korrectuly',
  'komrrectuy',
  'korrecbuy',
  'korrectucy',
  'korqectuy',
  'korrecutuy',
  'korrectuiy',
  'korrectiuy',
  'korrectyy',
  'korrectux',
  'korreqctuy',
  'korrrctuy',
  'korrectuhy',
  'kiorrectuy',
  'korrecjtuy',
  'kocrrectuy',
  'korresctuy',
  'koraectuy',
  'korrectu',
  'korrectyu',
  'korrectjuy',
  'korrefc

In [37]:
assert fix_typo_norvig("korrectud") == "corrected", "Norvig's correcter doesn't work"
assert fix_typo_norvig("speling") == "spelling", "Norvig's correcter doesn't work"
assert fix_typo_norvig("qwzxqwzxqwzx") == "qwzxqwzxqwzx"          # no candidate: returned unchanged
assert suggest_norvig("speling")[0] == "spelling"

# unit tests from Norvig's essay
for wrong, right in [("bycycle", "bicycle"), ("inconvient", "inconvenient"), ("arrainged", "arranged"),
                     ("peotry", "poetry"), ("peotryy", "poetry"), ("word", "word")]:
    assert fix_typo_norvig(wrong) == right, (wrong, fix_typo_norvig(wrong), right)
print("Norvig tests passed.")

Norvig tests passed.


#### Soundex

When the exact spelling is unknown, a **phonetic** algorithm lets the user type a word the way it sounds. Soundex maps a word to a 4-character code so that words that sound alike share the same code. Read *chapter 3.4* of the book. The version used here:

1. keep the first letter;
2. replace `a e i o u h w y` by `0`, `b f p v` by `1`, `c g j k q s x z` by `2`, `d t` by `3`, `l` by `4`, `m n` by `5`, `r` by `6`;
3. collapse **runs of the same digit** into one;
4. delete all zeros;
5. pad with trailing zeros / truncate to 4 characters (e.g. `k450`).

Candidates for a query are all dictionary words with the same code, ranked by edit distance (ties: more frequent first).

In [ ]:
SOUNDEX_GROUPS = ["aeiouhwy", "bfpv", "cgjkqsxz", "dt", "l", "mn", "r"]      # digit = position in list
SOUNDEX_DIGIT = {ch: str(d) for d, group in enumerate(SOUNDEX_GROUPS) for ch in group}


def produce_soundex_code(word):
    """Soundex code of a lowercase word (book version), e.g. 'britney' -> 'b635'."""
    word = "".join(c for c in word.lower() if c.isalpha())
    if not word:
        return ""
    
    # TODO
    
    return ''


def build_soundex_index(dictionary):
    """Map each Soundex code to the list of dictionary words that have it."""
    soundex_index = defaultdict(list)
    for word in dictionary:
        soundex_index[produce_soundex_code(word)].append(word)
    return dict(soundex_index)


def fix_typo_soundex(word, soundex_index, freqs=None) -> list:
    """Dictionary words with the same Soundex code, closest (edit distance) first.

    `freqs` (word -> count) is used to break ties; returns [word] if nothing matches.
    """
    matches = soundex_index.get(produce_soundex_code(word))
    if not matches:
        return [word]
    freqs = freqs or {}
    return sorted(matches, key=lambda w: (edit_dist(w, word), -freqs.get(w, 0), w))

#### Tests

In [39]:
soundex_index = build_soundex_index(vocabulary)

code1, code2 = produce_soundex_code("britney"), produce_soundex_code("breatany")
print(code1, code2)
assert code1 == code2

assert produce_soundex_code("robert") == produce_soundex_code("rupert") == "r163"
assert produce_soundex_code("anna") == "a500", "runs of equal digits must be collapsed"
assert produce_soundex_code("a") == "a000"

print(fix_typo_soundex("enouhg", soundex_index, vocabulary)[:10])
assert "enough" in fix_typo_soundex("enouhg", soundex_index, vocabulary), "Assert soundex failed"
print("Soundex tests passed.")

b635 b635
['enough', 'ensue', 'enjoy', 'enns', 'enwich', 'ewing', 'eying']
Soundex tests passed.


#### Trigrams with the Jaccard coefficient

Another view of a misspelled word is "a word that **shares many k-grams** with the typo". Using the k-gram index from Part 1 we collect all words that share at least one trigram with the query and rank them by the Jaccard coefficient

$$J(A, B) = \frac{|A \cap B|}{|A \cup B|},$$

where $A$ and $B$ are the sets of padded trigrams of the query and of the candidate word.

*Optimisation:* instead of building a trigram set for every candidate, we **count** in how many posting lists each word appears. That count is exactly $|A \cap B|$, and $|B|$ is looked up from a table computed once.

In [ ]:
def intersection_over_union(set1, set2):
    """Jaccard coefficient of two sets."""
    union = len(set1 | set2)
    return len(set1 & set2) / union if union else 0.0


def build_gram_counts(k_gram_index):
    """word -> number of distinct k-grams, i.e. |B| in the Jaccard formula."""
    sizes = Counter()

    # TODO
    
    return sizes


def fix_typo_kgram(word, k_gram_index, gram_counts=None, freqs=None) -> list:
    """Dictionary words ranked by Jaccard similarity of their k-grams to `word`.

    Ties are broken by word frequency (`freqs`), then alphabetically. Returns [word]
    when there is no candidate at all.
    """
    k = get_k(k_gram_index)
    word_grams = set(split_to_k_grams(f"${word}$", k))
    # TODO

    return []

#### Tests

In [40]:
# the optimised counting version must agree with the textbook definition
_a = set(split_to_k_grams("$enouh$", 3)); _b = set(split_to_k_grams("$enough$", 3))
assert abs(intersection_over_union(_a, _b) - len(_a & _b) / len(_a | _b)) < 1e-12

k_gram_index_vocab = build_k_gram_index(vocabulary, 3)
gram_counts_vocab = build_gram_counts(k_gram_index_vocab)

best = fix_typo_kgram("enouh", k_gram_index_vocab, gram_counts_vocab, vocabulary)
print(best[:20])
assert "enough" in best, "Assert k-gram failed"

for w in ["enouh", "speling", "recieve"]:
    ranked = fix_typo_kgram(w, k_gram_index_vocab, gram_counts_vocab, vocabulary)
    top = ranked[0]
    j_top = intersection_over_union(set(split_to_k_grams(f"${w}$", 3)), set(split_to_k_grams(f"${top}$", 3)))
    assert all(j_top >= intersection_over_union(set(split_to_k_grams(f"${w}$", 3)), set(split_to_k_grams(f"${c}$", 3))) - 1e-12
               for c in ranked[:200]), "ranking is not sorted by Jaccard"
assert fix_typo_kgram("", k_gram_index_vocab, gram_counts_vocab) == [""]
print("k-gram tests passed.")

['enough', 'ouh', 'venous', 'enormous', 'renounce', 'denounce', 'ravenous', 'denounced', 'en', 'saphenous', 'renounced', 'enormously', 'gangrenous', 'renouncing', 'denouncing', 'autogenous', 'denouement', 'indigenous', 'end', 'intravenous']
k-gram tests passed.


### 2.3 Estimate quality

We run each method on every `(right, wrong)` pair of the test sets and report

- **top-1 accuracy**: the first suggestion is the intended word (a word that is already in the dictionary is returned unchanged, so real-word errors count as failures);
- **top-5 accuracy** ("recall@5"): the intended word is among the first five suggestions (useful for a "Did you mean…?" list);
- the average **time per query**;
- the **ceiling**: the share of intended words that are in the vocabulary at all, i.e. the best any dictionary-based method can reach.

For reference, Norvig reports about 75 % (development) and 68 % (final) top-1 for his corrector, so the first row should be in that region; small differences come from tokenisation. Norvig's method suggests only words from the closest edit-distance tier, so it may list fewer than five words.

In [ ]:
methods = {
    "Norvig":         lambda w: suggest_norvig(w, 5),
    "Soundex":        lambda w: fix_typo_soundex(w, soundex_index, vocabulary)[:5],
    "k-gram/Jaccard": lambda w: fix_typo_kgram(w, k_gram_index_vocab, gram_counts_vocab, vocabulary)[:5],
}


def evaluate(pairs, suggest):
    """(top-1 accuracy, top-5 accuracy, ms per query) of `suggest` on (right, wrong) pairs."""
    top1 = top5 = 0
    start = time.perf_counter()
    for right, wrong in pairs:
        suggestions = # TODO
        top1 += # TODO
        top5 += # TODO
    elapsed = time.perf_counter() - start
    n = len(pairs)
    return top1 / n, top5 / n, 1000 * elapsed / n


for name, pairs in testsets.items():
    ceiling = sum(right in vocabulary for right, _ in pairs) / len(pairs)
    print(f"{name} set: {len(pairs)} test words, ceiling (intended word in vocabulary) = {ceiling:.1%}")
    print(f"  {'method':<16}{'top-1':>8}{'top-5':>8}{'ms/query':>11}")
    for method_name, suggest in methods.items():
        a1, a5, ms = evaluate(pairs, suggest)
        print(f"  {method_name:<16}{a1:>8.3f}{a5:>8.3f}{ms:>11.2f}")
    print()

development set: 270 test words, ceiling (intended word in vocabulary) = 94.4%
  method             top-1   top-5   ms/query
  Norvig             0.748   0.870      20.58
  Soundex            0.741   0.796       0.98
  k-gram/Jaccard     0.570   0.789       3.23

final set: 400 test words, ceiling (intended word in vocabulary) = 89.2%
  method             top-1   top-5   ms/query
  Norvig             0.675   0.780      22.41
  Soundex            0.657   0.760       1.03
  k-gram/Jaccard     0.605   0.748       3.08



### 2.4 Try it yourself

A tiny "did you mean" function that corrects every unknown word of a query with the method you choose. The title of the original version of this lab makes a good test.

In [20]:
def correct_query(query, method="norvig"):
    """Replace every out-of-vocabulary word of `query` by the best suggestion."""
    def fix(word):
        if word in vocabulary:
            return word
        if method == "norvig":
            return fix_typo_norvig(word)
        if method == "soundex":
            return fix_typo_soundex(word, soundex_index, vocabulary)[0]
        if method == "kgram":
            return fix_typo_kgram(word, k_gram_index_vocab, gram_counts_vocab, vocabulary)[0]
        raise ValueError(f"unknown method: {method}")
    return " ".join(fix(w) for w in sent_to_words(query))


query = "sorri not veri gud in inglish"
for method in ("norvig", "soundex", "kgram"):
    print(f"{method:<8} -> {correct_query(query, method)}")

norvig   -> sorry not very god in english
soundex  -> sorry not very god in incline
kgram    -> sorry not verify gun in english
